# Extraction accuracy check
Compares what the AI reads from the **11 real invoices** with the answers in `data/ground_truth/*.json`.
Use it to **choose the AI model** (accuracy vs speed) and to say a real number in the interview.

Steps: read PDF -> AI extraction -> clean -> compare field by field.

Known quirks of the answer files (not model errors): `SammyMaystoneLinesTest.json` has no amount, and for `oyo.json` the "invoice number" is a booking ID on a *receipt*, which our process correctly refuses to treat as an invoice.

## 0. Install packages (run once, then continue)
This installs into the same Python the notebook is using. If a package still isn't found afterwards, restart the kernel and run again.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, json, time
import pandas as pd
from rapidfuzz import fuzz

sys.path.insert(0, os.getcwd())          # run this notebook from the project folder
import pipeline as pl

USE_AI = True                            # False = rules only (shows how much the AI adds)
PROVIDER = "gemini"                      # "gemini" (needs GOOGLE_API_KEY) or "ollama"
MODEL = "gemini-3.7-flash"               # for ollama: "qwen2.5:3b"

pl.CONFIG["use_llm"] = USE_AI
pl.CONFIG["provider"] = PROVIDER
pl.CONFIG["model"] = MODEL
pl.CONFIG["use_cache"] = False
vendors, pos = pl.load_reference_data()

In [3]:
def load_truth(pdf_name):
    path = os.path.join(pl.DATA, "ground_truth", pdf_name.replace(".pdf", ".json"))
    with open(path) as f:
        data = json.load(f)
    return data[0] if isinstance(data, list) else data

def same_code(a, b):
    return pl.compact(a) == pl.compact(b) if a and b else False

def evaluate_one(pdf_path):
    name = os.path.basename(pdf_path)
    truth = load_truth(name)
    start = time.time()
    doc = pl.read_pdf(pdf_path)
    raw, method = pl.extract_fields(doc["text"], "eval_" + name, vendors["vendor_name"].tolist())
    c = pl.clean_fields(raw, doc["text"])
    seconds = time.time() - start

    gt_amount = truth.get("amount")
    return {
        "file": name,
        "vendor_ok": bool(c["vendor"]) and fuzz.token_set_ratio(c["vendor"], truth["issuer"]) >= 70,
        "number_ok": same_code(c["invoice_number"], truth.get("invoice_number")),
        "date_ok": c["invoice_date"] == truth.get("date"),
        "total_ok": (abs(c["total"] - gt_amount) < 0.01) if (c["total"] is not None and gt_amount is not None) else (None if gt_amount is None else False),
        "read": f"{c['vendor']} | {c['invoice_number']} | {c['invoice_date']} | {c['total']}",
        "expected": f"{truth['issuer']} | {truth.get('invoice_number')} | {truth.get('date')} | {gt_amount}",
        "method": method,
        "seconds": round(seconds, 1),
    }

In [4]:
folder = os.path.join(pl.DATA, "invoices", "real")
pdfs = sorted(os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".pdf"))
results = pd.DataFrame([evaluate_one(p) for p in pdfs])
results

c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-fl

,file,vendor_ok,number_ok,date_ok,total_ok,read,expected,method,seconds
0,AmazonWebServices.pdf,True,True,True,True,"Amazon Web Services, Inc. | 42183017 | 2014-08...",Amazon Web Services | 42183017 | 2014-08-03 | ...,AI (gemini-3.6-flash),28.4
1,AzureInterior.pdf,False,True,True,True,Global Wholesaler | INV/2023/03/0008 | 2023-03...,Azure Interior | INV/2023/03/0008 | 2023-03-20...,AI (gemini-3.5-flash-lite),7.5
2,FlipkartInvoice.pdf,False,True,True,True,WS Retail Services Pvt. Ltd. | BLR_WFLD2015100...,Flipkart | #BLR_WFLD20151000982590 | 2015-10-2...,AI (gemini-3.5-flash-lite),43.4
3,NetpresseInvoice.pdf,False,True,True,True,ALEXINUX | 2022089083 | 2022-11-28 | 56.02,NETPRESSE | 2022089083 | 2022-11-28 | 56.02,AI (gemini-3.5-flash-lite),18.3
4,QualityHosting.pdf,True,True,True,True,QualityHosting AG | 30064443 | 2014-05-07 | 34.73,QualityHosting AG | 30064443 | 2014-05-07 | 34.73,AI (gemini-3.5-flash-lite),63.5
5,SammyMaystoneLinesTest.pdf,True,True,True,None,Sammy Maystone | invoice_number_1 | 2022-01-01...,Sammy Maystone - For Lines testing | invoice_n...,AI (gemini-3.5-flash-lite),33.5
6,coolblue1.pdf,True,True,True,True,Coolblue B.V. | 993548900 | 2014-04-19 | 717.97,Coolblue B.V. | 993548900 | 2014-04-19 | 717.97,AI (gemini-3.5-flash-lite),4.7
7,coolblue2.pdf,True,True,True,True,Coolblue B.V. | 992288600 | 2014-03-29 | 4904.94,Coolblue B.V. | 992288600 | 2014-03-29 | 4904.94,AI (gemini-3.5-flash-lite),9.1
8,free_fiber.pdf,True,True,True,True,Free Service Abonné | 562044387 | 2015-07-02 |...,Free | 562044387 | 2015-07-02 | 29.99,AI (gemini-3.5-flash-lite),7.7
9,oyo.pdf,False,False,True,True,Oravel Stays Pvt. Ltd. | None | 2017-12-31 | 1...,OYO | IBZY2087 | 2017-12-31 | 1939.0,AI (gemini-3.5-flash-lite),32.9


In [5]:
ai_failed = results["method"].str.startswith("rules")
if USE_AI and ai_failed.any():
    print(f"WARNING: the AI did NOT answer for {int(ai_failed.sum())} of {len(results)} invoices, so these numbers are rules-only.")
    print("Run  python check_setup.py  in the terminal to find out why.\n")

fields = ["vendor_ok", "number_ok", "date_ok", "total_ok"]
summary = {f.replace("_ok", ""): f"{int(results[f].fillna(False).sum())}/{int(results[f].notna().sum())}" for f in fields}
print("Correct fields:", summary)
print("Average time per invoice:", round(results["seconds"].mean(), 1), "seconds")
results[results[fields].eq(False).any(axis=1)][["file", "read", "expected"]]

Correct fields: {'vendor': '7/11', 'number': '10/11', 'date': '11/11', 'total': '10/10'}
Average time per invoice: 23.1 seconds


,file,read,expected
1,AzureInterior.pdf,Global Wholesaler | INV/2023/03/0008 | 2023-03...,Azure Interior | INV/2023/03/0008 | 2023-03-20...
2,FlipkartInvoice.pdf,WS Retail Services Pvt. Ltd. | BLR_WFLD2015100...,Flipkart | #BLR_WFLD20151000982590 | 2015-10-2...
3,NetpresseInvoice.pdf,ALEXINUX | 2022089083 | 2022-11-28 | 56.02,NETPRESSE | 2022089083 | 2022-11-28 | 56.02
9,oyo.pdf,Oravel Stays Pvt. Ltd. | None | 2017-12-31 | 1...,OYO | IBZY2087 | 2017-12-31 | 1939.0


## Compare models
Pull each model first (`ollama pull <name>`), then run this cell. `ai_answered` must equal 11, otherwise the AI did not run and the accuracy numbers are rules-only. Pick the model with the best accuracy that is still fast enough for a live demo.

In [6]:
comparison = []
for model in [MODEL]:                              # add more model names to compare
    pl.CONFIG["model"] = model
    rows = pd.DataFrame([evaluate_one(p) for p in pdfs])
    comparison.append({"model": model,
                       "vendor": int(rows["vendor_ok"].sum()), "number": int(rows["number_ok"].sum()),
                       "date": int(rows["date_ok"].sum()), "total": int(rows["total_ok"].fillna(False).sum()),
                       "ai_answered": int((~rows["method"].str.startswith("rules")).sum()),
                       "avg_seconds": round(rows["seconds"].mean(), 1)})
pd.DataFrame(comparison)

c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\adity\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3

,model,vendor,number,date,total,ai_answered,avg_seconds
0,gemini-3.7-flash,8,10,11,10,11,5.3
